# 外部candidate dataset: raw scan / token inspection

現在のrawファイルをread-onlyで検査し、raw → preprocessing → prompt → local tokenizerを確認します。
このNotebook単独で上から実行できます。もう一方のデータ調査は [01_data_and_token_inspection.ipynb](01_data_and_token_inspection.ipynb) に分離しています。
data/の変更・ダウンロード・モデル推論・API呼び出しは行いません。結果はメモリ上に保持します。

全体scan → 一覧 → 保存inventoryとの任意比較 → 選択record → 対象範囲のtoken計測・比較の順です。


## 1. imports / path

scan対象directory、実ファイルのschemaを取得する関数、reader、preprocessing、token counterを明示します。


In [ ]:
from pathlib import Path
import sys
import json
import pandas as pd
from IPython.display import display
ROOT = next((p for p in [Path.cwd(), *Path.cwd().parents]
             if (p / "dataset_adapters").is_dir() and (p / "configs/sources.json").is_file()), None)
if ROOT is None:
    raise RuntimeError("workspace直下またはnotebooks/から起動してください。")
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

from tools.inventory_candidates import scan
from dataset_adapters.base import inventory_context
from dataset_adapters.registry import get_adapter, EXTERNAL
from tools.live_inspection import (
    external_frames, compare_frames, show_frame,
    token_summary_frame, compare_saved_tokens,
)
from tools.notebook_inspection import (
    select_record, preview_raw, preprocessing_for_inspection,
    selected_query, token_record, read_batch_summary,
)
from tools.count_input_tokens import resolve_encoding
from tools.token_inventory import build_token_inventory

pd.set_option("display.max_columns", None)
pd.set_option("display.max_rows", 30)
print("External raw:", ROOT / "data/candidate_datasets")
print("No API clients. Scan/token results stay in memory; saved inventories are not overwritten.")


## 2. LIVE SCAN: external raw

scan()が現在取得済みの全ファイルを再帰scanします。
CSV/TXT/MAT/ULog/XLSXは既存の安全なreaderで確認し、code/model/media等はmetadataのみです。
opaque MATLAB objectは未解釈のまま示します。scan内の個別ファイルerrorも後の表に残し、成功扱いしません。


In [ ]:
candidate_inventory = scan(ROOT / "data/candidate_datasets")
print("LIVE external files:", len(candidate_inventory["files"]))
print("Source size/mtime unchanged:", candidate_inventory["source_size_mtime_unchanged"])
print("Read errors:", sum(f["status"] == "error" for f in candidate_inventory["files"]))


## 3. external全file / schema / column shape

candidate_inventory objectから作成。external_files_dfは全取得物、external_columns_dfはcolumn schema、
external_arrays_dfはMAT/NPZ/ULogの配列shape/dtypeです。CSVのdtypeはinventory readerの型解釈で、ファイル固有のbinary dtypeではありません。

外部は数万ファイルあるため、表示だけページに分けます。全件は各DataFrameに保持しています。
EXTERNAL_FILE_START / EXTERNAL_COLUMN_START / EXTERNAL_ARRAY_STARTを変更してこのセルを再実行するか、query()で絞り込めます。
missingが未報告の形式はnullであり0を捏造しません。


In [ ]:
external_datasets_df, external_files_df, external_schema_groups_df, external_columns_df, external_arrays_df = external_frames(candidate_inventory)
EXTERNAL_FILE_START = 0
EXTERNAL_COLUMN_START = 0
EXTERNAL_ARRAY_START = 0
PAGE_SIZE = 100
show_frame(external_datasets_df, title="LIVE external_datasets_df")
show_frame(external_files_df, title="LIVE external_files_df", start=EXTERNAL_FILE_START, page_size=PAGE_SIZE)
show_frame(external_schema_groups_df, title="LIVE external_schema_groups_df")
show_frame(external_columns_df, title="LIVE external_columns_df", start=EXTERNAL_COLUMN_START, page_size=PAGE_SIZE)
show_frame(external_arrays_df, title="LIVE external_arrays_df", start=EXTERNAL_ARRAY_START, page_size=PAGE_SIZE)
scan_errors_df = external_files_df.query("status == 'error'")
if not scan_errors_df.empty:
    display(scan_errors_df)


## 4. LIVE SCAN vs SAVED INVENTORY — secondary validation

ここで初めて保存済みinventoryを読みます。ファイルがない場合もlive inspectionは可能です。
全件のkeyでouter joinし、新規／削除／shape・dtype変更・件数差を隠しません。
一致しなくてもrawを変更したりcacheを上書きしたりしません。


In [ ]:
saved_base = ROOT / "outputs/dataset_inventory"
raw_validation = {}
if (saved_base / "candidate_schema.json").is_file():
    saved_candidates = json.loads((saved_base / "candidate_schema.json").read_text())
    _, saved_external_files, _, saved_external_columns, saved_external_arrays = external_frames(saved_candidates)
    raw_validation["external_files"] = compare_frames(
        external_files_df, saved_external_files, ["dataset", "relative_path"],
        ["size_bytes", "rows", "columns", "schema_group", "status"])
    raw_validation["external_columns"] = compare_frames(
        external_columns_df, saved_external_columns, ["dataset", "schema_group", "column"], ["dtype", "rows", "missing"])
    raw_validation["external_arrays"] = compare_frames(
        external_arrays_df, saved_external_arrays, ["dataset", "relative_path", "key"], ["shape", "dtype", "ndim", "opaque"])
    display(pd.DataFrame([{"comparison": name, "rows": len(table),
                          "matches": int(table.matches.sum()), "differences": int((~table.matches).sum())}
                         for name, table in raw_validation.items()]))
    for name, table in raw_validation.items():
        if not table.matches.all():
            show_frame(table.loc[~table.matches], title=name + " differences", page_size=30)
else:
    print("Saved inventory not found. LIVE scan results remain the primary source.")


## 5. drill-down selection

全体を見た後、ここでdatasetを選びます。番号は0-basedです。
選択を変更したら「selected raw record」から「selected query token」まで再実行してください。
外部のみ選択可能: f16capstone / trajair / aircombat_wez / calculated_moves / baidu_fighter_jet（未取得）。
staticのROW_IDは全reader単位の累積IDです。前処理は本projectのexample / baselineであり、raw datasetそのものではありません。


In [ ]:
DATASET = "f16capstone"
EPISODE = 0
SEQUENCE_ID = None
ROW_ID = None
INDEX = 10

HISTORY_SIZE = 1
METRIC = None
QUESTION_NAME = None  # 外部baselineでは通常None。
MODEL = "gpt-4o"
# 外部は明示した既存設定のみ。例: "configs/preprocessing/f16capstone_default.yaml"
PREPROCESSING_CONFIG = "configs/preprocessing/f16capstone_default.yaml"


## 6. selected raw record — RAW DATA

readerには同じlive inventoryを渡します。保存済みcacheをprimary sourceへ戻しません。
source file、選択sequence、全raw fieldのshape/dtypeと実値を表示します。


In [ ]:
if DATASET not in EXTERNAL:
    raise ValueError("このNotebookの対象外datasetです。もう一方のNotebookを使用してください。")
with inventory_context({"tasks": []}, candidate_inventory):
    adapter = get_adapter(DATASET)
sequence, raw_record = select_record(adapter, episode=EPISODE, sequence_id=SEQUENCE_ID, row_id=ROW_ID, index=INDEX)
if sequence is None:
    print("NOT_ACQUIRED: raw inspection unavailable.")
else:
    print("RAW RECORD")
    print("Source:", sequence.source_file, "\nSequence:", sequence.sequence_id, sequence.key)
    print("Episode:", raw_record["episode_id"], "Row:", raw_record.get("row_id"), "Index:", raw_record["index"])
    print("SHA256:", raw_record["source_file_sha256"])
    raw_fields_df = pd.DataFrame([{"field": name, **spec} for name, spec in raw_record["fields"].items()])
    show_frame(raw_fields_df, title="Selected raw fields")
    print(json.dumps(raw_record["fields"], ensure_ascii=False, indent=2))
    raw_preview = preview_raw(adapter, sequence)
    if isinstance(raw_preview, dict):
        for key, value in raw_preview.items():
            print("\n", key, "(episode scope)" if key == "episodic_return" else "[:5]")
            print(value)
    else:
        display(raw_preview)


## 7. PREPROCESSING → HISTORY

RAWとMODEL INPUT AFTER PREPROCESSINGは別です。
外部は指定YAMLのexample / baseline変換です。raw datasetそのものではありません。存在しないreward/actionの生成や欠損値の0埋めはしません。


In [ ]:
config, preprocessing_status = preprocessing_for_inspection(DATASET, PREPROCESSING_CONFIG)
print(preprocessing_status)
query = None
if config is not None and sequence is not None:
    display(config)
    try:
        query = selected_query(adapter, sequence, raw_record["index"], config, HISTORY_SIZE, METRIC, QUESTION_NAME)
    except (ValueError, IndexError) as exc:
        print("Model input: N/A —", exc)
    if query is not None:
        print("MODEL INPUT AFTER PREPROCESSING:", config["prompt_mode"])
        print("Metric / question:", query.metric, query.question_name)
        print("H:", query.history_size, "history [start,end):", query.history_start, query.history_end)
        print("\nHISTORY\n", query.history_text)
else:
    print("Token count: N/A")


## 8. actual prompt全文

同じpreprocessingが作ったSYSTEM PROMPTとUSER PROMPT。文字列を表示するだけで送信しません。


In [ ]:
if query is not None:
    print("Prompt mode:", config["prompt_mode"], "Preprocessing:", config["name"])
    print("\nSYSTEM PROMPT\n")
    print(query.system_prompt)
    print("\nUSER PROMPT\n")
    print(query.user_prompt)
else:
    print("Prompt: N/A")


## 9. selected query token

CLI共通count_queryを使用。contentは実文字列のlocal tokenize結果、API入力はcontent+9の既存概算でserver usageではありません。
history/questionはBPE境界のためuser_tokensに単純加算できません。fallback使用時は明示します。


In [ ]:
selected_tokens = None
if query is not None:
    encoding, fallback = resolve_encoding(MODEL)
    selected_tokens = token_record(query, sequence, config, encoding, fallback, MODEL)
    print("Model:", MODEL, "Tokenizer:", encoding.name, "Fallback:", fallback)
    names = ["system_tokens", "history_tokens", "question_tokens", "user_tokens",
             "total_content_tokens", "estimated_api_input_tokens"]
    display(pd.DataFrame([{"component": k, "tokens": selected_tokens[k]} for k in names]))
else:
    print("Token count: N/A")


## 10. LIVE TOKEN COUNT: external datasets

build_token_inventory()をCLIと共有し、今回の実行でprompt文字列を作りlocal tokenizeします。選択recordとは独立です。
外部のみを対象に既存default YAMLを使用。100,000件超のpoolは既定2,000件・seed=0でsampling。staticはH=0。sample総tokenは全件総tokenではありません。未取得・未対応は理由付きskip。
結果はmemoryに保持し、保存済みbatchを上書きしません。


In [ ]:
TOKEN_MODEL = "gpt-4o"
TOKEN_HISTORY_SIZES = [1]
TOKEN_EXTERNAL_FULL = False
TOKEN_EXTERNAL_SAMPLE_SIZE = 2000
TOKEN_EXTERNAL_SAMPLE_THRESHOLD = 100000
TOKEN_SEED = 0


In [ ]:
with inventory_context({"tasks": []}, candidate_inventory):
    result = build_token_inventory(
        datasets="external-all", history_sizes=TOKEN_HISTORY_SIZES, model=TOKEN_MODEL,
        external_full=TOKEN_EXTERNAL_FULL,
        external_sample_size=TOKEN_EXTERNAL_SAMPLE_SIZE,
        external_sample_threshold=TOKEN_EXTERNAL_SAMPLE_THRESHOLD,
        seed=TOKEN_SEED,
    )
print("LIVE count:", result.metadata["number_of_datasets"], "datasets,",
      result.metadata["number_of_queries"], "query/history/static rows")
print("Tokenizer:", result.metadata["tokenizer"], "Fallback:", result.metadata["fallback_encoding_used"])
token_scopes_df = pd.DataFrame(result.scopes)
token_skipped_df = pd.DataFrame(result.skipped)
show_frame(token_scopes_df, title="LIVE scopes — full selected scope vs deterministic sample")
show_frame(token_skipped_df, title="Skipped datasets and reasons")


## 11. all_token_queries_df

1 query / history window / static record=1行。全計測行をmemory上のDataFrameとして保持します。
画面はページ表示だけですが、all_token_queries_df.query(...)で全件を直接検索できます。
episode_id / sequence_id / row_idは非該当ならNAです。


In [ ]:
all_token_queries_df = pd.DataFrame(result.queries)
for col in ["episode_id", "sequence_id", "row_id", "query_index"]:
    all_token_queries_df[col] = all_token_queries_df[col].astype("Int64")
QUERY_START = 0
query_columns = [
    "dataset_id", "source_file", "episode_id", "sequence_id", "row_id", "query_index",
    "history_size", "prompt_mode", "preprocessing_name", "preprocessing_config_sha256",
    "estimated_api_input_tokens", "system_tokens", "user_tokens", "history_tokens", "question_tokens",
    "count_scope", "eligible_query_pool", "sample_size", "seed",
]
show_frame(all_token_queries_df[query_columns], title="LIVE all_token_queries_df", start=QUERY_START, page_size=100)
# 例: all_token_queries_df.query("dataset_id == 'f16capstone' and sequence_id == 0")
token_by_sequence_df = pd.DataFrame(result.by_sequence)


## 12. token_by_dataset_df

dataset / mode / H / 前処理設定ごとに分離します。異なるprompt semanticsやsample scopeを混ぜて合算しません。
stdは母標準偏差、p95はnearest-rank（既存CLIと共通）です。


In [ ]:
token_by_dataset_df = token_summary_frame(result)
summary_columns = [
    "dataset_id", "prompt_mode", "preprocessing_name", "history_size", "count_scope",
    "eligible_query_pool", "sample_size", "seed", "counted_queries", "mean", "std",
    "min", "median", "p95", "max", "total",
]
show_frame(token_by_dataset_df[summary_columns], title="LIVE token_by_dataset_df")


## 13. dataset間token distribution

prompt_modeごとに図を分け、各datasetのpreprocessing_name・H・全件/sampleをラベルに表示します。
boxplotのひげは1.5×IQR、点はその外側。幅や点の数から未計測poolの分布を推定しません。


In [ ]:
import matplotlib.pyplot as plt
for mode, mode_df in all_token_queries_df.groupby("prompt_mode", sort=True):
    grouped = list(mode_df.groupby(["dataset_id", "preprocessing_name", "history_size", "count_scope"], sort=True))
    labels = [f"{key[0]}\n{key[1]} | H={key[2]} | {key[3]}" for key, _ in grouped]
    values = [group["estimated_api_input_tokens"].to_numpy() for _, group in grouped]
    fig, ax = plt.subplots(figsize=(13, max(3, 0.65 * len(grouped))))
    ax.boxplot(values, vert=False)
    ax.set_yticks(range(1, len(labels) + 1), labels)
    ax.set_xlabel("Estimated API input tokens (content + 9)")
    ax.set_title(f"Prompt mode: {mode}")
    fig.tight_layout()
    plt.show()


## 14. largest/smallest token queries → raw/prompt drill-down

以下の行を見て「drill-down selection」の選択値を変更してください。
前処理設定・H・metric/questionも一致させ、raw表示から選択queryのtoken計測まで再実行すると同じpromptを確認できます。
各query行に実効設定JSONとSHA256を保持しています。


In [ ]:
largest_token_queries_df = all_token_queries_df.nlargest(20, "estimated_api_input_tokens")
smallest_token_queries_df = all_token_queries_df.nsmallest(20, "estimated_api_input_tokens")
drill_columns = query_columns + ["metric", "question_name"]
show_frame(largest_token_queries_df[drill_columns], title="Largest 20 counted queries")
show_frame(smallest_token_queries_df[drill_columns], title="Smallest 20 counted queries")
example = largest_token_queries_df.iloc[0]
print("Example selection — copy into drill-down selection:")
print("DATASET =", repr(example["dataset_id"]))
print("EPISODE =", int(example["episode_id"]) if pd.notna(example["episode_id"]) else None)
print("SEQUENCE_ID =", int(example["sequence_id"]))
print("ROW_ID =", int(example["row_id"]) if pd.notna(example["row_id"]) else None)
print("INDEX =", int(example["query_index"]))
print("HISTORY_SIZE =", int(example["history_size"]))
print("Metric/question:", example["metric"], example["question_name"])
print("Effective preprocessing JSON:", example["preprocessing_config"])


## 15. optional: Saved batch resultsとのvalidation

read_batch_summary()はここだけで使用します。LIVE結果がprimaryです。
同じdataset・mode・H・設定SHA・model/tokenizer・sampling条件の集計値を比較し、savedのみのHも明示します。
比較は集計値レベルです。prompt全文の厳密一致を調べる場合はquery-levelのprompt SHAを照合してください。
savedがなければ表示するだけで、追加batchは起動しません。


In [ ]:
saved_token_summary = read_batch_summary()
saved_metadata_path = ROOT / "outputs/token_counts/summary.json"
if saved_token_summary is None or not saved_metadata_path.is_file():
    print("Token batch has not been executed yet.")
else:
    saved_token_summary = saved_token_summary.loc[saved_token_summary["dataset_id"].isin(EXTERNAL)].copy()
    saved_metadata = json.loads(saved_metadata_path.read_text())
    saved_token_validation_df = compare_saved_tokens(result, saved_token_summary, saved_metadata)
    show_frame(saved_token_validation_df, title="LIVE vs SAVED token aggregates")
    print("Matched aggregates:", int(saved_token_validation_df.matches.sum()))
    print("Saved results read only; no CSV/JSON was overwritten.")
